In [1]:
from pathlib import Path
from collections import Counter
import wfdb
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "data" / "raw"

print("Project:", PROJECT_ROOT)
print("Data:", DATA_ROOT)

Project: d:\Sanika_EGM_Project
Data: d:\Sanika_EGM_Project\data\raw


In [2]:
dataset_candidates = list(DATA_ROOT.rglob("ARGODataset_Folder"))

print("ARGO dataset folders found:", len(dataset_candidates))

for folder in dataset_candidates:
    print(folder)

ARGO dataset folders found: 1
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder


In [3]:
ARGO_ROOT = dataset_candidates[0]

print("Using:")
print(ARGO_ROOT)

Using:
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder


In [4]:
annotation_files = list(ARGO_ROOT.rglob("*.annotation_*"))

print("Total annotation files:", len(annotation_files))

for file in annotation_files[:30]:
    print(file)

Total annotation files: 7848
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1000.annotation_ann1
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1000.annotation_ann2
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1000.annotation_ann3
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1000.annotation_consensu

In [5]:
annotation_types = sorted(
    set(file.suffixes[-1].replace(".", "") for file in annotation_files)
)

print(annotation_types)

['annotation_ann1', 'annotation_ann2', 'annotation_ann3', 'annotation_consensus']


In [6]:
annotation_extensions = sorted(
    set(file.name.split(".", 1)[1] for file in annotation_files)
)

print("Annotation types found:")
for ext in annotation_extensions:
    print(ext)

Annotation types found:
annotation_ann1
annotation_ann2
annotation_ann3
annotation_consensus


In [7]:
consensus_files = [
    f for f in annotation_files
    if ".annotation_consensus" in f.name
]

print("Consensus files:", len(consensus_files))

for f in consensus_files[:10]:
    print(f)

Consensus files: 1962
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1000.annotation_consensus
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1002.annotation_consensus
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1003.annotation_consensus
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1004.annotation_

In [8]:
consensus_file = consensus_files[0]

print("Selected:")
print(consensus_file)

Selected:
d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1000.annotation_consensus


In [9]:
record_name = consensus_file.stem.split(".")[0]
annotation_extension = consensus_file.name.split(".", 1)[1]

print("Record:", record_name)
print("Annotation extension:", annotation_extension)

Record: P1000
Annotation extension: annotation_consensus


In [10]:
annotation = wfdb.rdann(
    str(consensus_file.parent / record_name),
    extension=annotation_extension
)

print("Annotation loaded successfully!")
print("Number of annotations:", len(annotation.sample))
print("First samples:", annotation.sample[:20])
print("First symbols:", annotation.symbol[:20])

Annotation loaded successfully!
Number of annotations: 2
First samples: [2022 2107]
First symbols: ['(', ')']


In [11]:
symbol_counts = Counter(annotation.symbol)

print("Unique annotation symbols:")
for symbol, count in symbol_counts.most_common():
    print(f"{repr(symbol)} : {count}")

Unique annotation symbols:
'(' : 1
')' : 1


In [12]:
annotation_table = pd.DataFrame({
    "sample": annotation.sample,
    "symbol": annotation.symbol
})

print(annotation_table.head(30))

   sample symbol
0    2022      (
1    2107      )


In [13]:
print(annotation_table.shape)

(2, 2)


In [14]:
print("Number of aux notes:", len(annotation.aux_note))

for i in range(min(30, len(annotation.aux_note))):
    print(
        "sample =", annotation.sample[i],
        "| symbol =", annotation.symbol[i],
        "| aux_note =", repr(annotation.aux_note[i])
    )

Number of aux notes: 2
sample = 2022 | symbol = ( | aux_note = 'A'
sample = 2107 | symbol = ) | aux_note = 'A'


In [15]:
def load_annotation(file_path):
    record_name = file_path.stem.split(".")[0]
    extension = file_path.name.split(".", 1)[1]

    return wfdb.rdann(
        str(file_path.parent / record_name),
        extension=extension
    )

In [16]:
for ext in [
    "annotation_ann1",
    "annotation_ann2",
    "annotation_ann3",
    "annotation_consensus"
]:
   
    files = [
        f for f in annotation_files
        if f.name.endswith("." + ext)
    ]
   
    if not files:
        print(ext, "NOT FOUND")
        continue
   
    ann = load_annotation(files[0])
   
    print("\n==============================")
    print(ext)
    print("==============================")
    print("File:", files[0])
    print("Number of annotations:", len(ann.sample))
    print("Symbols:", Counter(ann.symbol))


annotation_ann1
File: d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1000.annotation_ann1
Number of annotations: 2
Symbols: Counter({'(': 1, ')': 1})

annotation_ann2
File: d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1000.annotation_ann2
Number of annotations: 2
Symbols: Counter({'(': 1, ')': 1})

annotation_ann3
File: d:\Sanika_EGM_Project\data\raw\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\annotated-dataset-of-post-ischemic-ventricular-tachycardia-electrograms-argo-1.0.0\ARGODataset_Folder\Pt1\P1000.annotation_ann3
Number of annotations: 2
Symbols: Counter({'(': 1, ')': 1})

annotation_consensus
Fil

In [17]:
summary_rows = []

for ext in [
    "annotation_ann1",
    "annotation_ann2",
    "annotation_ann3",
    "annotation_consensus"
]:
   
    files = [
        f for f in annotation_files
        if f.name.endswith("." + ext)
    ]
   
    for file in files:
        ann = load_annotation(file)
       
        counts = Counter(ann.symbol)
       
        for symbol, count in counts.items():
            summary_rows.append({
                "file": file.name,
                "annotation_type": ext,
                "symbol": symbol,
                "count": count
            })

annotation_summary = pd.DataFrame(summary_rows)

print(annotation_summary.head())
print("\nRows:", len(annotation_summary))

                    file  annotation_type symbol  count
0  P1000.annotation_ann1  annotation_ann1      (      1
1  P1000.annotation_ann1  annotation_ann1      )      1
2  P1002.annotation_ann1  annotation_ann1      (      1
3  P1002.annotation_ann1  annotation_ann1      )      1
4  P1003.annotation_ann1  annotation_ann1      "      1

Rows: 11773


In [18]:
annotation_summary.to_csv(
    PROJECT_ROOT / "results" / "annotation_summary.csv",
    index=False
)

print("Saved annotation summary.")

Saved annotation summary.
